# LSTMs from Scratch (and why vanilla RNNs forget)

**Learning objectives**
- Diagnose vanishing gradients on a **long-range** copy task
- Implement LSTM gates and cell state updates in NumPy
- Watch LSTM succeed where a vanilla RNN struggles
- Map the equations to PyTorch `nn.LSTM`

Run cells top-to-bottom. Constants are grouped near the top so you can experiment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# --- Hyperparameters (tweak these) ---
HIDDEN_SIZE = 24
DELAY = 12          # how far the signal must travel
SEQ_LEN = DELAY + 4
LEARNING_RATE = 0.05
EPOCHS = 120
BATCH_SIZE = 64
N_TRAIN = 800
N_TEST = 200

## 1. Long-range dependency task

**Copy-first token:** each sequence is

$$
\big[\, s,\; \underbrace{0,0,\ldots,0}_{\text{DELAY blanks}},\; \texttt{GO}\,\big]
$$

and the label is $s\in\{1,2,3\}$. The model must remember $s$ across `DELAY` distractor steps until `GO`.

Vanilla RNNs often fail as `DELAY` grows; LSTMs are designed for this.

In [ ]:
# Vocabulary: 0=blank, 1..3=signal, 4=GO
BLANK, GO = 0, 4
V = 5


def make_copy_task(n, delay=DELAY, rng=None):
    rng = rng or np.random.default_rng(0)
    T = delay + 2  # signal, blanks..., GO
    X = np.zeros((n, T), dtype=np.int64)
    y = rng.integers(1, 4, size=n)  # signal tokens 1,2,3
    X[:, 0] = y
    X[:, -1] = GO
    # middle already blank
    return X, y


X_train, y_train = make_copy_task(N_TRAIN, rng=np.random.default_rng(0))
X_test, y_test = make_copy_task(N_TEST, rng=np.random.default_rng(1))
SEQ_LEN = X_train.shape[1]
print(f'seq_len={SEQ_LEN}, delay={DELAY}')
print('example X[0]:', X_train[0], ' y=', y_train[0])

## 2. LSTM equations

An LSTM maintains a **cell state** $c_t$ (long-term memory) and hidden state $h_t$ (readout).

Gates (all in $(0,1)$ via sigmoid $\sigma$):

$$
\begin{aligned}
f_t &= \sigma(W_f[x_t; h_{t-1}] + b_f) & &\text{forget} \\
i_t &= \sigma(W_i[x_t; h_{t-1}] + b_i) & &\text{input} \\
o_t &= \sigma(W_o[x_t; h_{t-1}] + b_o) & &\text{output} \\
\tilde{c}_t &= \tanh(W_c[x_t; h_{t-1}] + b_c) & &\text{candidate}
\end{aligned}
$$

State updates:

$$
c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t
$$

$$
h_t = o_t \odot \tanh(c_t)
$$

**Learning note:** If $f_t \approx 1$ and $i_t \approx 0$, then $c_t \approx c_{t-1}$ — a **constant error carousel**. Gradients can flow across many steps without repeated $\tanh'$ shrinkage.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -20, 20)))


def softmax(Z):
    Z = Z - np.max(Z, axis=1, keepdims=True)
    e = np.exp(Z)
    return e / e.sum(axis=1, keepdims=True)


def one_hot_seq(X, v=V):
    N, T = X.shape
    out = np.zeros((N, T, v))
    out[np.arange(N)[:, None], np.arange(T)[None, :], X] = 1.0
    return out


def one_hot_y(y, k=3):
    # map labels 1,2,3 -> columns 0,1,2
    Y = np.zeros((len(y), k))
    Y[np.arange(len(y)), y - 1] = 1.0
    return Y


def init_lstm(h=HIDDEN_SIZE, v=V, rng=np.random.default_rng(42)):
    # 4 gates share input dim v+h
    din = v + h
    scale = 0.1
    W = rng.normal(0, scale, size=(din, 4 * h))
    b = np.zeros((1, 4 * h))
    # forget-gate bias trick: start with remembering
    b[0, :h] = 1.0
    Why = rng.normal(0, scale, size=(h, 3))
    by = np.zeros((1, 3))
    return {'W': W, 'b': b, 'Why': Why, 'by': by}


def lstm_forward(X_ids, params):
    X = one_hot_seq(X_ids)
    N, T, v = X.shape
    h = HIDDEN_SIZE
    hs = [np.zeros((N, h))]
    cs = [np.zeros((N, h))]
    gates = []  # store f,i,g,o, pre-activations for backprop
    for t in range(T):
        xh = np.concatenate([X[:, t], hs[-1]], axis=1)  # (N, v+h)
        pre = xh @ params['W'] + params['b']            # (N, 4h)
        f = sigmoid(pre[:, :h])
        i = sigmoid(pre[:, h:2*h])
        g = np.tanh(pre[:, 2*h:3*h])   # candidate
        o = sigmoid(pre[:, 3*h:])
        c = f * cs[-1] + i * g
        ht = o * np.tanh(c)
        gates.append((xh, pre, f, i, g, o, cs[-1], c))
        cs.append(c)
        hs.append(ht)
    logits = hs[-1] @ params['Why'] + params['by']
    probs = softmax(logits)
    cache = (X, hs, cs, gates, probs)
    return probs, cache


params = init_lstm()
p0, _ = lstm_forward(X_train[:2], params)
print('init probs', np.round(p0, 3))

## 3. Backprop through LSTM (high level)

From $c_t = f_t \odot c_{t-1} + \cdots$ we get:

$$
\frac{\partial L}{\partial c_{t-1}} \;\mathrel{+}=\; \frac{\partial L}{\partial c_t} \odot f_t
$$

When forget gates stay open ($f_t\sim 1$), **cell gradients pass almost unchanged**. That is the architectural fix for long delays.

Below: full NumPy BPTT for our gate packing, then a matching vanilla RNN for comparison.

In [ ]:
def lstm_backward(y, params, cache):
    X, hs, cs, gates, probs = cache
    N, T, v = X.shape
    h = HIDDEN_SIZE
    Y = one_hot_y(y)
    dlogits = (probs - Y) / N

    grads = {k: np.zeros_like(val) for k, val in params.items()}
    grads['Why'] = hs[-1].T @ dlogits
    grads['by'] = dlogits.sum(axis=0, keepdims=True)

    dh_next = dlogits @ params['Why'].T
    dc_next = np.zeros((N, h))

    for t in reversed(range(T)):
        xh, pre, f, i, g, o, c_prev, c = gates[t]
        # h = o * tanh(c)
        do = dh_next * np.tanh(c)
        dc = dh_next * o * (1 - np.tanh(c) ** 2) + dc_next

        df = dc * c_prev
        di = dc * g
        dg = dc * i
        dc_prev = dc * f

        # backprop through activations
        dpre_f = df * f * (1 - f)
        dpre_i = di * i * (1 - i)
        dpre_g = dg * (1 - g ** 2)
        dpre_o = do * o * (1 - o)
        dpre = np.concatenate([dpre_f, dpre_i, dpre_g, dpre_o], axis=1)

        grads['W'] += xh.T @ dpre
        grads['b'] += dpre.sum(axis=0, keepdims=True)

        dxh = dpre @ params['W'].T
        dh_next = dxh[:, v:]  # part w.r.t. previous h
        dc_next = dc_prev

    return grads


def clip_grads(grads, max_norm=5.0):
    total = np.sqrt(sum(np.sum(g ** 2) for g in grads.values()))
    if total > max_norm:
        for k in grads:
            grads[k] *= max_norm / (total + 1e-8)
    return grads


def loss_acc(probs, y):
    Y = one_hot_y(y)
    eps = 1e-8
    loss = float(-np.mean(np.sum(Y * np.log(np.clip(probs, eps, 1)), axis=1)))
    pred = np.argmax(probs, axis=1) + 1
    acc = float(np.mean(pred == y))
    return loss, acc

In [ ]:
# Vanilla RNN baseline (same task)
def init_rnn(h=HIDDEN_SIZE, v=V, rng=np.random.default_rng(0)):
    s = 0.1
    return {
        'Wxh': rng.normal(0, s, (v, h)),
        'Whh': rng.normal(0, s, (h, h)),
        'bh': np.zeros((1, h)),
        'Why': rng.normal(0, s, (h, 3)),
        'by': np.zeros((1, 3)),
    }


def rnn_forward(X_ids, params):
    X = one_hot_seq(X_ids)
    N, T, _ = X.shape
    h = HIDDEN_SIZE
    hs = [np.zeros((N, h))]
    pretanh = []
    for t in range(T):
        z = X[:, t] @ params['Wxh'] + hs[-1] @ params['Whh'] + params['bh']
        pretanh.append(z)
        hs.append(np.tanh(z))
    probs = softmax(hs[-1] @ params['Why'] + params['by'])
    return probs, (X, hs, pretanh, probs)


def rnn_backward(y, params, cache):
    X, hs, pretanh, probs = cache
    N, T, _ = X.shape
    Y = one_hot_y(y)
    dlogits = (probs - Y) / N
    grads = {k: np.zeros_like(v) for k, v in params.items()}
    grads['Why'] = hs[-1].T @ dlogits
    grads['by'] = dlogits.sum(0, keepdims=True)
    dh = dlogits @ params['Why'].T
    for t in reversed(range(T)):
        dz = dh * (1 - np.tanh(pretanh[t]) ** 2)
        grads['Wxh'] += X[:, t].T @ dz
        grads['Whh'] += hs[t].T @ dz
        grads['bh'] += dz.sum(0, keepdims=True)
        dh = dz @ params['Whh'].T
    return grads

## 4. Train RNN vs LSTM side by side

In [ ]:
def train_model(kind, X, y, X_te, y_te, epochs=EPOCHS, lr=LEARNING_RATE, batch=BATCH_SIZE):
    if kind == 'lstm':
        params = init_lstm()
        fwd, bwd = lstm_forward, lstm_backward
    else:
        params = init_rnn()
        fwd, bwd = rnn_forward, rnn_backward
    rng = np.random.default_rng(1)
    hist = []
    n = len(y)
    for epoch in range(epochs):
        idx = rng.permutation(n)
        for start in range(0, n, batch):
            bi = idx[start:start + batch]
            probs, cache = fwd(X[bi], params)
            grads = clip_grads(bwd(y[bi], params, cache))
            for k in params:
                params[k] -= lr * grads[k]
        te_p, _ = fwd(X_te, params)
        _, te_acc = loss_acc(te_p, y_te)
        hist.append(te_acc)
        if (epoch + 1) % 30 == 0 or epoch == 0:
            print(f'{kind:4s} epoch {epoch+1:3d}  test={te_acc*100:.1f}%')
    return params, hist


print(f'Comparing on DELAY={DELAY} (seq_len={SEQ_LEN})')
_, hist_rnn = train_model('rnn', X_train, y_train, X_test, y_test)
lstm_params, hist_lstm = train_model('lstm', X_train, y_train, X_test, y_test)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([a * 100 for a in hist_rnn], label='vanilla RNN', linewidth=2)
ax.plot([a * 100 for a in hist_lstm], label='LSTM', linewidth=2)
ax.axhline(100 / 3, color='gray', ls='--', label='chance (33%)')
ax.set_xlabel('epoch'); ax.set_ylabel('test accuracy %')
ax.set_title(f'Long-range copy task (delay={DELAY})')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

### Pause & Reflect — LSTM

1. What does a forget-gate bias of $+1$ encourage at initialization?
2. Why can $\partial L/\partial c_{t-1}$ stay large even when $\tanh'$ on $h$ is small?
3. If you reduce `DELAY` to 2, do you still expect a big RNN–LSTM gap?

### Discussion — LSTM

1. **Forget bias**: $\sigma(1)\approx 0.73$ — start closer to "remember" than "erase".
2. **Cell path**: Gradients through $c$ multiply by $f_t$, not by $\tanh'$ of the hidden readout — a highway past saturating nonlinearities.
3. **Short delay**: Vanilla RNNs often suffice; the LSTM advantage appears as the dependency lengthens.

## 5. Inspect forget-gate activity

After training, average $f_t$ across the blank region should stay high if the cell is holding the signal.

In [ ]:
# Reuse trained lstm_params; collect forget gates on a test batch
X = one_hot_seq(X_test[:64])
N, T, v = X.shape
h = HIDDEN_SIZE
ht = np.zeros((N, h))
ct = np.zeros((N, h))
forget_traj = []
for t in range(T):
    xh = np.concatenate([X[:, t], ht], axis=1)
    pre = xh @ lstm_params['W'] + lstm_params['b']
    f = sigmoid(pre[:, :h])
    i = sigmoid(pre[:, h:2*h])
    g = np.tanh(pre[:, 2*h:3*h])
    o = sigmoid(pre[:, 3*h:])
    ct = f * ct + i * g
    ht = o * np.tanh(ct)
    forget_traj.append(f.mean())

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(forget_traj, marker='o')
ax.set_xlabel('time step'); ax.set_ylabel('mean forget-gate value')
ax.set_title('Forget gate stays open across blanks (memory highway)')
ax.set_ylim(0, 1); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 6. PyTorch comparison

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

class CopyLSTM(nn.Module):
    def __init__(self, v=V, h=HIDDEN_SIZE):
        super().__init__()
        self.embed = nn.Embedding(v, h)
        self.lstm = nn.LSTM(h, h, batch_first=True)
        self.fc = nn.Linear(h, 3)
    def forward(self, x):
        out, (h_n, c_n) = self.lstm(self.embed(x))
        return self.fc(h_n.squeeze(0))


model = CopyLSTM()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
crit = nn.CrossEntropyLoss()
# labels 1,2,3 -> 0,1,2 for CE
y_tr = torch.tensor(y_train - 1)
y_te = torch.tensor(y_test - 1)
loader = DataLoader(TensorDataset(torch.tensor(X_train), y_tr),
                    batch_size=BATCH_SIZE, shuffle=True)
X_te_t = torch.tensor(X_test)

torch_accs = []
for epoch in range(EPOCHS):
    model.train()
    for xb, yb in loader:
        opt.zero_grad()
        loss = crit(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
    model.eval()
    with torch.no_grad():
        pred = model(X_te_t).argmax(1)
        torch_accs.append((pred == y_te).float().mean().item())
    if (epoch + 1) % 30 == 0 or epoch == 0:
        print(f'PyTorch LSTM epoch {epoch+1:3d}  test={torch_accs[-1]*100:.1f}%')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot([a * 100 for a in hist_lstm], label='NumPy LSTM', linewidth=2)
ax.plot([a * 100 for a in torch_accs], label='PyTorch LSTM', linewidth=2)
ax.set_xlabel('epoch'); ax.set_ylabel('test accuracy %')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Summary

| Piece | Role |
|-------|------|
| Forget gate $f_t$ | How much old $c$ to keep |
| Input gate $i_t$ | How much new candidate to write |
| Output gate $o_t$ | How much of $\tanh(c)$ to expose as $h$ |
| Cell state $c_t$ | Additive memory path → long-range credit assignment |

**Next:** Transformers drop recurrence and use **attention** to mix tokens in parallel.